In [1]:
import torch

In [2]:
# Dans PyTorch, le tenseur est la brique de base universelle (building block). Tout est uniformisé sous forme de tenseur, qu'il s'agisse :
#  - d'un simple nombre (scalaire)
#  - d'une liste de nombres (vecteur)
#  - d'une image (matrice 2D)
#  - d'une vidéo (tenseur 4D)
# simples nombres emballés dans un tenseur, 
# ce qui leur permet de profiter de la machinerie de PyTorch (graphe de calcul, compatibilité GPU, etc.) tout en restant un scalaire.
x, y = torch.tensor(2.), torch.tensor(7.)

In [5]:
# requires_grad (booléen) indique à PyTorch s'il doit suivre les opérations faites sur ce tenseur 
#  pour calculer automatiquement les gradients plus tard.# 
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [7]:
# Loss function : Elle quantifie l'erreur de votre modèle.
#  Elle mesure à quel point la prédiction du modèle s'éloigne de la vraie valeur cible (y).
#  Squared Error
L = (w * x + b - y) ** 2

L.backward() lance la rétropropagation du gradient (backpropagation).       
- Elle calcule les dérivées (gradients) : En partant de la perte L, elle parcourt le graphe de calcul à l'envers 
  (grâce à la règle de dérivation en chaîne / chain rule) pour mesurer l'impact de chaque paramètre (w et b) sur l'erreur commise.
- Elle stocke les résultats : Elle enregistre ces gradients directement dans l'attribut .grad des tenseurs concernés 
  (c'est-à-dire ceux qui ont l'attribut requires_grad=True).

PyTorch calcule deux dérivées : comment $L$ varie si on bouge un tout petit peu $w$, et comment $L$ varie si on bouge un tout petit peu $b$.

Pour le poids $w$ ($\frac{\partial L}{\partial w}$) :$$\frac{\partial L}{\partial w} = 2 \cdot (w \cdot x + b - y) \cdot x$$

Pour le biais $b$ ($\frac{\partial L}{\partial b}$) :$$\frac{\partial L}{\partial b} = 2 \cdot (w \cdot x + b - y) \cdot 1$$

In [ ]:
# Une fois ces calculs de dérivées terminés, PyTorch va ranger ces résultats directement là où il faut :
# - il stocke -20. dans l'attribut w.grad.
# - il stocke -10. dans l'attribut b.grad.
L.backward()
print(w.grad, b.grad)

tensor(-20.) tensor(-10.)


##### Une fois les gradients calcules, PyTorch détruit le graphe de calcul de l'exemple L = (w * x + b - y) et libère toute la mémoire associée. 
- Si on décompose cela sous forme de graphe, PyTorch ne fait pas tout en un seul bloc. Il découpe le calcul en petites étapes successives :
    - Étape 1 (Multiplication) : On multiplie w par x $\rightarrow$ cela donne un résultat intermédiaire (disons $A = w \times x$).
    - Étape 2 (Addition) : On ajoute b à ce résultat $A$ $\rightarrow$ cela donne la prédiction ($B = A + b$, ou $\hat{y}$).
    - Étape 3 (Soustraction) : On soustrait y à cette prédiction $\rightarrow$ cela donne l'écart ($C = B - y$).
    - Étape 4 (Puissance) : On élève cet écart $C$ au carré $\rightarrow$ cela donne la perte finale (L).

- Forward pass : pendant le trajet aller, PyTorch avance de gauche à droite (de w, x, b, y vers L) en enregistrant chaque maillon de la chaîne dans sa mémoire. Il construit le graphe.
- Retropropragation : pendant le trajet retour, PyTorch repart de la fin (L) et remonte le graphe à l'envers, de droite à gauche, en appliquant la règle de dérivation en chaîne (chain rule) sur chaque petit maillon pour propager l'erreur jusqu'à w et b.

In [ ]:
# Si les gradients existent, on les remet à zéro
if w.grad is not None:
    w.grad.zero_()
if b.grad is not None:
    b.grad.zero_()

In [22]:
L = (w * x + b - y) ** 2

retain_graph=False
L.backward(retain_graph=retain_graph)
print(w.grad, b.grad)

tensor(-20.) tensor(-10.)
